### Transformer

In [ ]:
import numpy, pandas, sklearn, torch, transformers
print("numpy        :", numpy.__version__)
print("pandas       :", pandas.__version__)
print("scikit-learn :", sklearn.__version__)
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)

In [ ]:
from pathlib import Path

GOEMOTIONS_CSVS = [
    "archive (7)/data/full_dataset/goemotions_1.csv",
    "archive (7)/data/full_dataset/goemotions_2.csv",
    "archive (7)/data/full_dataset/goemotions_3.csv",
]
GOEMO_TEXT_COL  = "text"      

ARTEMIS_CSV     = "official_data/artemis_caption_label.csv"
ART_TEXT_COL    = "caption"
ART_LABEL_COL   = "label"

# ---- Training and mapping params ----
BASE_MODEL = "distilbert-base-uncased"  
EPOCHS     = 2                        
BATCH_SIZE = 8                         
LR         = 3e-5
VAL_SIZE   = 0.1
SEED       = 42
MAX_LEN    = 128                  

OUT_GOEMO  = Path("./out_goemo_1")   # trained classifier
OUT_MAP    = Path("./out_map_1")     # learned mapping + heatmaps
OUT_GOEMO.mkdir(parents=True, exist_ok=True)
OUT_MAP.mkdir(parents=True, exist_ok=True)

ARTEMIS_9 = ['something else','sadness','contentment','awe','amusement','excitement','fear','disgust','anger']


In [ ]:
import pandas as pd

src = "official_data/artemis_dataset_release_v0.csv" 
df = pd.read_csv(src)
# columns: 'utterance' (text), 'emotion' (label)
out = df[['utterance','emotion']].rename(columns={'utterance':'caption','emotion':'label'})
out['label'] = (out['label'].astype(str).str.strip().str.lower()
                .str.replace('_',' ', regex=False).str.replace('-',' ', regex=False)
                .str.replace(r'\s+',' ', regex=True))
out.to_csv("official_data/artemis_caption_label.csv", index=False)
print(out.head(), "\nSaved -> official_data/artemis_caption_label.csv")

In [ ]:
import numpy as np
import json

M = np.load(OUT_MAP / "mapping_A1.npy")  

def to_artemis_percent(text: str):
    # get 27-way probs from our trained model
    out = (lambda o: {d["label"]: float(d["score"]) for d in o})(
        __import__("transformers").pipelines.pipeline(
            "text-classification",
            model=str(OUT_GOEMO / "model"),
            tokenizer=str(OUT_GOEMO / "model"),
            top_k=None,
            function_to_apply="sigmoid",
            device=-1
        )(text)[0]
    )
    labels = json.loads((OUT_GOEMO / "label_list.json").read_text())
    p27 = np.array([out.get(l, 0.0) for l in labels], dtype=np.float32)
    p27 = p27 / (p27.sum() + 1e-12)
    p9 = p27 @ M
    p9 = p9 / (p9.sum() + 1e-12)

    result = {
        "probabilities": {emo: float(p) for emo, p in zip(ARTEMIS_9, p9)},
        # "percentages": {emo: float(100*p) for emo, p in zip(ARTEMIS_9, p9)}
    }

    # print nicely
    print(json.dumps(result, indent=4))

    return result


print(to_artemis_percent("I will destroy you and anyone you have ever met!"))

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline
from transformers import pipeline, CLIPTextModel, CLIPTokenizer

# -------------------------
# CONFIG
# -------------------------
device = "cpu"             
OUT_DIR = Path("emotion_sd_lora_final")
OUT_DIR.mkdir(exist_ok=True)
OUT_GOEMO = Path("out_goemo_1")   
OUT_MAP = Path("out_map_1")      

# ARTEMIS labels
ARTEMIS_9 = ['something else','sadness','contentment','awe','amusement','excitement','fear','disgust','anger']

# -------------------------
# Sanity checks / loads
# -------------------------
assert (OUT_GOEMO / "model").exists(), f"Cannot find classifier at {OUT_GOEMO / 'model'}"
assert (OUT_MAP / "mapping_A1.npy").exists(), f"Cannot find mapping at {OUT_MAP / 'mapping_A1.npy'}"

print("Loading GoEmotions classifier (pipeline)...")
classifier = pipeline(
    "text-classification",
    model=str(OUT_GOEMO / "model"),
    tokenizer=str(OUT_GOEMO / "model"),
    top_k=None,
    function_to_apply="sigmoid",
    device=-1  # CPU pipeline
)

M_A1 = np.load(OUT_MAP / "mapping_A1.npy")

### Diffusion

In [ ]:
# -------------------------
# Load Stable Diffusion pipeline (text encoder inside)
# -------------------------
sd_model_id = "runwayml/stable-diffusion-v1-5"
print("Loading Stable Diffusion pipeline (this may take a moment)...")
pipe = StableDiffusionPipeline.from_pretrained(
    sd_model_id,
    torch_dtype=torch.float32,
    safety_checker=None
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()
print("Pipeline loaded. Device:", device)

tokenizer = pipe.tokenizer

# -------------------------
# LoRA wrapper
# -------------------------
class LoRALinear(nn.Module):
    def __init__(self, linear: nn.Linear, r: int = 8, alpha: float = 8.0):
        super().__init__()
        self.orig = linear
        self.r = r
        self.alpha = alpha
        if r > 0:
            self.lora_down = nn.Linear(linear.in_features, r, bias=False)
            self.lora_up = nn.Linear(r, linear.out_features, bias=False)
            nn.init.normal_(self.lora_down.weight, std=0.01)
            nn.init.zeros_(self.lora_up.weight)
        else:
            self.lora_down = None
            self.lora_up = None
        # freeze original linear parameters
        for p in self.orig.parameters():
            p.requires_grad = False
        # compute scale
        self.scale = float(alpha) / max(1, r)

        # Ensure this wrapper has parameters on same device as orig
        self.to(next(self.orig.parameters()).device)

    def forward(self, x):
        base = self.orig(x)
        if self.r > 0:
            return base + self.lora_up(self.lora_down(x)) * self.scale
        else:
            return base

def apply_lora_recursively(module: nn.Module, r: int = 8, target_types=(nn.Linear,)):
    """
    Replace Linear modules recursively with LoRALinear wrappers.
    Keeps module structure otherwise intact.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, target_types):
            wrapped = LoRALinear(child, r=r, alpha=8.0)
            setattr(module, name, wrapped)
        else:
            apply_lora_recursively(child, r=r, target_types=target_types)

In [ ]:
# -------------------------
# Inference: generate images using classifier -> mapping -> prompt
# -------------------------
print("\nGenerating images using the LoRA-enabled pipeline (may be slow on CPU)...")

def text_to_artemis_probs(text):
    """
    Converts a sentence to a 9-dimensional ARTEMIS probability vector.
    Includes a manual adjustment to map high-arousal positive emotions
    (approval, joy) more strongly to excitement.
    """
    # raw GoEmotions 27-class scores
    res = classifier(text)[0]
    labels27 = json.loads((OUT_GOEMO / "label_list.json").read_text())
    score_dict = {r['label']: r['score'] for r in res}
    p27 = np.array([score_dict.get(l, 0.0) for l in labels27], dtype=np.float32)
    
    if p27.sum() > 0:
        p27 /= p27.sum()
    else:
        p27 = np.ones_like(p27) / len(p27)
    
    # map 27 → 9
    p9 = p27 @ M_A1

    idx_approval = labels27.index("approval")
    idx_joy      = labels27.index("joy")
    idx_contentment = ARTEMIS_9.index("contentment")
    idx_excitement  = ARTEMIS_9.index("excitement")
    
    p9[idx_excitement] += p27[idx_approval]*0.7 + p27[idx_joy]*0.7
    p9[idx_contentment] -= p27[idx_approval]*0.5 + p27[idx_joy]*0.5

    # clip & normalize
    p9 = np.clip(p9, 1e-6, None)
    p9 /= p9.sum()
    
    return p9


emotion_style = {
    "anger": "furious volcanic eruption of rage, dark crimson and black abstract expressionism, "
             "Franz Kline and de Kooning style, violent slashing brush strokes, raw emotional fury, "
             "dramatic chiaroscuro, museum-quality oil painting masterpiece",
    
    "disgust": "grotesque decaying flesh and organic matter, sickly green-yellow pus, "
               "visceral revulsion, Francis Bacon distorted bodies, hyper-detailed surreal horror, "
               "nauseating atmosphere, museum-quality oil painting",
    
    "fear": "primordial terror in the dark, shadowy figures emerging from mist, "
             "Zdzisław Beksiński dystopian nightmare, eerie blue moonlight, creeping dread, "
             "psychological horror masterpiece, museum-quality oil painting",
    
    "sadness": "profound melancholy, lone figure in pouring rain under streetlight, "
               "Edward Hopper solitude, deep blues and muted grays, tears on face, "
               "heartbreaking cinematic atmosphere, museum-quality oil painting",
    
    "contentment": "warm golden hour glow over peaceful meadow, soft pastel skies, "
                   "Claude Monet impressionism, gentle breeze, pure tranquility and inner peace, "
                   "serene masterpiece, museum-quality oil painting",
    
    "excitement": "explosive joy and celebration, massive colorful fireworks bursting across the night sky, "
                  "vibrant neon colors, dynamic Futurist energy, electric pop-art style, "
                  "pure exhilaration and life force, high-energy masterpiece",
    
    "awe": "transcendent cosmic wonder, majestic snow-capped mountains bathed in divine golden light rays, "
           "Caspar David Friedrich romanticism, sublime overwhelming scale, "
           "breathtaking spiritual beauty, museum-quality oil painting",
    
    "amusement": "playful whimsical delight, colorful carnival of laughter and joy, "
                 "Henri Matisse vibrant colors, dancing figures with big smiles, "
                 "lighthearted fun and innocent happiness, cheerful masterpiece",
    
    "something else": "ethereal abstract meditation, soft pastel color field gradients, "
                      "Mark Rothko contemplative calm, emotional ambiguity, "
                      "pure abstract expression, no figures, serene and mysterious"
}

sentences = [
    "I will destroy you and anyone you have ever met!",
    "I had a good time with my friends this weekend",
    "My heart is heavy after hearing the news of the loss of my dog.",
    "Oh my God!!! We won the game!!",
    "The spoilt milk in the back of the fridge made me so nauseous.."
]

for i, s in enumerate(sentences, 1):
    p9 = text_to_artemis_probs(s)
    dominant = ARTEMIS_9[int(np.argmax(p9))]
    confidence = p9.max()
    style = emotion_style[dominant]  

    if dominant in ["excitement", "amusement"]:
        prefix = "vibrant emotionally intense masterpiece, highly detailed, dramatic lighting, "
    elif dominant == "something else":
        prefix = "pure abstract expression, "
    else:
        prefix = "museum quality oil painting with visible brush strokes, highly detailed, dramatic lighting, "
    
    prompt = f"{prefix}expressing intense {dominant}, {style}, emotional masterpiece, 8k resolution"

    print(f"\n[{i}/5] Sent: {s}")
    print(f"[{i}/9] Detected: {dominant.upper():12}")
    #print("  Detected dominant:", dominant)
    print("  Prompt (truncated):", prompt[:120])

    # Generate
    with torch.no_grad():
        out = pipe(prompt, num_inference_steps=25, guidance_scale=7.5, height=512, width=512)
        image = out.images[0]

    fname = OUT_DIR / f"final_art_enc_{i}_{dominant}.png"
    image.save(fname)
    print("  Saved:", fname)

print("\nFinished. Results saved to:", OUT_DIR)

In [ ]:
# ================================
# SIMPLE GRADIO INTERFACE
# ================================
import gradio as gr
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import time

def generate(sentence):
    # 1. Emotion classification
    res = classifier(sentence)[0]
    
    with open(OUT_GOEMO / "label_list.json") as f:
        labels = json.load(f)
        
    p27 = np.array([next((r['score'] for r in res if r['label']==l), 0) for l in labels])
    p27 /= p27.sum() + 1e-8
    
    # 27 → 9
    p27 += 1e-6
    p27 /= p27.sum()
    p9 = p27 @ M_A1
    p9 += 1e-6
    p9 /= p9.sum()
    dominant = ARTEMIS_9[np.argmax(p9)]
    
    style = emotion_style.get(dominant, emotion_style["something else"])
    if dominant in ["excitement", "amusement"]:
        prefix = "vibrant emotionally intense masterpiece, highly detailed, dramatic lighting, "
    elif dominant == "something else":
        prefix = "pure abstract expression, "
    else:
        prefix = "museum quality oil painting with visible brush strokes, highly detailed, dramatic lighting, "
    
    
    # creative card HTML
    emotion_html = f"""
    <div style="
        background: linear-gradient(135deg, #6a11cb, #2575fc);
        padding: 20px;
        border-radius: 15px;
        color: white;
        font-size: 26px;
        text-align: center;
        font-weight: bold;
        box-shadow: 0px 4px 15px rgba(0,0,0,0.3);
        margin-top: 20px;
    ">
        <div>✨ Detected Emotion ✨</div>
        <div style="font-size:40px; margin-top:10px;">{dominant.upper()}</div>
    </div>
    """
    
    # Yield emotion immediately
    yield None, emotion_html
    
    # rotating “while painting” messages
    msgs = [
        "Mixing colors from your feelings… 🎨",
        "Letting your soul guide the brush… ✨",
        "Shaping emotion into texture… 🌌",
        "Painting what words cannot say… 💫",
        "Almost there — beauty takes time… ⏳"
    ]
    
    # show rotating messages while generating
    for m in msgs:
        yield None, emotion_html + f"<p style='text-align:center; color:#666;'>{m}</p>"
        time.sleep(1.2)
    
    prompt = f"{prefix}expressing intense {dominant}, {style}, emotional masterpiece, 8k resolution"

    with torch.no_grad():
        image = pipe(prompt, num_inference_steps=28, guidance_scale=8.0).images[0]

    # final output
    yield image, emotion_html


# ======================== GRADIO INTERFACE ========================
with gr.Blocks(
    theme=gr.themes.Soft(),
    css="""
    .title {font-size: 48px; font-weight: 900;
            background: linear-gradient(90deg,#ff7eb3,#ff758c,#ff7eb3);
            -webkit-background-clip:text; -webkit-text-fill-color:transparent;
            text-align:center; margin-bottom:10px;}
    .subtitle {text-align:center; font-size:18px; color:#444;}
    .loading-box {text-align:center; font-size:20px; padding:20px; color:#555;}
    """
) as demo:

    gr.HTML("<h1 class='title'>Emotion → Art</h1>")
    gr.HTML("<p class='subtitle'>Your feelings, transformed into a masterpiece.</p>")
    
    with gr.Row():
        with gr.Column():
            textbox = gr.Textbox(label="Speak your emotion", lines=4)
            btn = gr.Button("Paint My Soul 🎨", variant="primary")
        with gr.Column():
            img = gr.Image(label="Generated Artwork", height=512)
    
    emotion_html = gr.HTML("")
    
    btn.click(generate, inputs=textbox, outputs=[img, emotion_html])

print("Launching your emotional art gallery...")
demo.launch(share=True, server_name="0.0.0.0", server_port=7875)  ## -> please change the port number if it shows already in use